# 州×商品カテゴリから販促候補を考える

## このNotebookの問い

PB・APはSPより平均注文額が高い。一方で注文数は少ない。各州で何が購入されているかを確認し、販促候補を絞る前に必要な情報を整理する。

ここで分かるのは過去の注文構成であり、広告を出した場合の追加売上や利益ではない。

**指標の定義**

- 州の平均注文額 = 州のGMV ÷ 州のユニークな注文数
- カテゴリ購入割合 = そのカテゴリを含む注文数 ÷ 州のユニークな注文数
- 州内カテゴリGMVシェア = そのカテゴリのGMV ÷ 州のGMV

1注文に複数カテゴリがある場合、カテゴリ購入割合の合計は100%を超えることがある。

In [1]:
import pandas as pd
from IPython.display import display

# 既存のAthena集計結果を読み込む。
df_baseline = pd.read_csv("../data/sales_baseline_by_month_state.csv")
df_category = pd.read_csv("../data/sales_by_category_state.csv")

# 比較したい州を明示する。SPは規模の大きい比較対象。
target_states = ["SP", "PB", "AP"]

In [2]:
# 月×州の集計を州単位に戻す。注文数はカテゴリごとの件数を足さない。
state_totals = (
    df_baseline.groupby("customer_state", as_index=False)
    .agg(state_orders=("order_count", "sum"), state_gmv=("gmv", "sum"))
)
state_totals["average_order_value"] = (
    state_totals["state_gmv"] / state_totals["state_orders"]
)

display(
    state_totals.loc[
        state_totals["customer_state"].isin(target_states),
        ["customer_state", "state_orders", "state_gmv", "average_order_value"],
    ].round(2)
)

,customer_state,state_orders,state_gmv,average_order_value
3,AP,67,13374.81,199.62
14,PB,517,112586.82,217.77
25,SP,40501,5067633.16,125.12


In [4]:
# 州の注文数とGMVをカテゴリ別の行に付け、分母をそろえる。
category_comparison = (
    df_category.loc[df_category["customer_state"].isin(target_states)]
    .merge(state_totals, on="customer_state", how="left")
)
category_comparison["category_order_rate_pct"] = (
    category_comparison["category_order_count"]
    / category_comparison["state_orders"] * 100
)
category_comparison["category_gmv_share_pct"] = (
    category_comparison["category_gmv"]
    / category_comparison["state_gmv"] * 100
)

# 各州でGMV上位5カテゴリを表示し、件数と割合を並べて読む。
top_categories = (
    category_comparison.sort_values(
        ["customer_state", "category_gmv"], ascending=[True, False]
    )
    .groupby("customer_state", group_keys=False)
    .head(5)
)
display(
    top_categories[
        ["customer_state", "product_category", "category_order_count",
         "category_order_rate_pct", "category_gmv", "category_gmv_share_pct"]
    ].round(2)
)

,customer_state,product_category,category_order_count,category_order_rate_pct,category_gmv,category_gmv_share_pct
0,AP,computers_accessories,6,8.96,2050.03,15.33
1,AP,watches_gifts,7,10.45,1639.80,12.26
2,AP,computers,1,1.49,1437.00,10.74
3,AP,health_beauty,10,14.93,1380.58,10.32
4,AP,auto,3,4.48,1216.39,9.09
21,PB,health_beauty,75,14.51,15692.08,13.94
22,PB,watches_gifts,41,7.93,13102.98,11.64
23,PB,computers_accessories,40,7.74,7195.08,6.39
24,PB,computers,4,0.77,7128.00,6.33
25,PB,garden_tools,14,2.71,5590.01,4.97


## 結果を読む際の確認事項

1. PB・APの上位カテゴリは、何件の注文に支えられているか。少数の高額注文に偏っていないか。
2. SPと同じカテゴリでも、購入割合とGMVシェアはどう違うか。
3. この集計だけで需要の不足や広告の効果を断定しない。次に必要な情報は、期間別の安定性、商品・販売者の構成、広告費・利益に関する情報。

結果を確認してから、どのカテゴリを詳しく調べるかを決める。配送フォロー施策については、別途レビュー・注文時系列の分析条件を整理する。